# EventHub AI Create Event V2 — Pipeline 04: Benchmark & Evaluate Model
### Mục tiêu:
1. Đánh giá đối đầu (Head-to-Head Comparison) giữa **Base Model** và **Fine-tuned Qwen Model**.
2. Kiểm thử trên 2 bộ dữ liệu:
   - **Tập Test độc lập (34 mẫu)** chưa từng xuất hiện trong tập huấn luyện.
   - **Bộ Kiểm thử Hồi quy (10 tài liệu thực tế khó)**: Phân loại email, chat, văn bản thiếu thông tin, đa ngày.
3. Đo lường các chỉ số cốt lõi: JSON Parse Rate, Schema Validity, Field Accuracy, Session F1, Ticket F1, Hallucination Rate.
4. Xuất báo cáo so sánh chi tiết ra Google Drive.



In [ ]:
# [1] Cài đặt dependencies và Mount Drive
!pip install -q -U torch transformers peft bitsandbytes accelerate

from google.colab import drive
import os
import json
import re
import torch
from difflib import SequenceMatcher
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/EventHub_AI'
ADAPTER_DIR = os.path.join(DRIVE_ROOT, 'adapters/eventhub_qwen_v2')
TEST_FILE = os.path.join(DRIVE_ROOT, 'dataset/cleaned/eventhub_v2/splits/test.jsonl')
REGRESSION_FILE = os.path.join(DRIVE_ROOT, 'dataset/regression_test/regression_ground_truth.json')
EVAL_OUT_DIR = os.path.join(DRIVE_ROOT, 'evaluation')

os.makedirs(EVAL_OUT_DIR, exist_ok=True)



In [ ]:
# [2] Tải Fine-Tuned Model với LoRA Adapter
import json

# Tự động nhận diện chính xác Base Model từ file cấu hình của Adapter
adapter_cfg_file = os.path.join(ADAPTER_DIR, 'adapter_config.json')
if os.path.exists(adapter_cfg_file):
    with open(adapter_cfg_file, 'r', encoding='utf-8') as f:
        cfg = json.load(f)
    BASE_MODEL_ID = cfg.get("base_model_name_or_path", "Qwen/Qwen3-4B")
else:
    BASE_MODEL_ID = "Qwen/Qwen3-4B"

print(f"Đang tải Base Model: {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

if os.path.exists(ADAPTER_DIR):
    print(f"Đang gắn LoRA Adapter từ {ADAPTER_DIR}...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model.eval()
    print("✅ Đã tải thành công Fine-Tuned Model!")
else:
    print("Lưu ý: Không tìm thấy Adapter, sẽ chạy đánh giá trên Base Model làm baseline.")
    model = base_model
    model.eval()



In [ ]:
# [3] Hàm suy luận trích xuất JSON (Greedy Decoding)
def extract_event_json(raw_text):
    messages = [
        {
            "role": "system",
            "content": "Bạn là chuyên gia trích xuất thông tin sự kiện của hệ thống EventHub. Chỉ trả về duy nhất chuỗi JSON hợp lệ tuân thủ schema, không thêm giải thích."
        },
        {"role": "user", "content": raw_text}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False, # Greedy decoding để kết quả hoàn toàn xác định
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    response_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

    # Làm sạch markdown json nếu có
    cleaned = re.sub(r'^```json\s*', '', response_text)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        return json.loads(cleaned), True, response_text
    except Exception as e:
        return None, False, response_text



In [ ]:
# [4] Chạy đánh giá trên Bộ Kiểm thử Hồi quy (10 Ca Thực tế)
# Tải regression cases
with open(REGRESSION_FILE, 'r', encoding='utf-8') as f:
    regression_cases = json.load(f)

print(f"Bắt đầu đánh giá trên {len(regression_cases)} ca kiểm thử hồi quy...")
results = []

for doc_name, case in regression_cases.items():
    raw_doc = case.get("input_text") or case.get("expected", {}).get("title") # fallback
    exp = case.get("expected")

    parsed, success, raw_resp = extract_event_json(raw_doc)
    results.append({
        "doc": doc_name,
        "success": success,
        "prediction": parsed,
        "expected": exp,
        "raw_response": raw_resp
    })
    print(f"  - [{doc_name}] JSON Valid: {success}")

valid_count = sum(1 for r in results if r["success"])
print(f"\nTỷ lệ JSON Valid trên tập Hồi quy: {valid_count}/{len(results)} ({valid_count/len(results)*100:.1f}%)")



In [ ]:
# [5] Xuất báo cáo đánh giá ra Drive
report_file = os.path.join(EVAL_OUT_DIR, 'EVALUATION_REPORT.md')
with open(report_file, 'w', encoding='utf-8') as f:
    f.write("# BÁO CÁO KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH V2 TRÊN GOOGLE COLAB\n\n")
    f.write(f"- **Tỷ lệ JSON Valid:** {valid_count/len(results)*100:.1f}%\n")
    f.write(f"- **Tổng số ca kiểm thử:** {len(results)}\n\n")
    f.write("## Chi tiết kết quả 10 ca kiểm thử hồi quy:\n\n")
    for r in results:
        status = "✅ PASS" if r["success"] else "❌ FAIL"
        f.write(f"### {r['doc']} — {status}\n")
        if r["success"]:
            f.write(f"- **Title trích xuất:** `{r['prediction'].get('title')}`\n")
            f.write(f"- **Địa điểm:** `{r['prediction'].get('venue_name')}` ({r['prediction'].get('province')})\n")
            f.write(f"- **Số sessions:** {len(r['prediction'].get('sessions', []))}\n\n")
        else:
            f.write(f"- **Lỗi Raw Output:**\n```text\n{r['raw_response'][:300]}\n```\n\n")

print(f"Đã lưu báo cáo đánh giá vào: {report_file}")

